# 02 · Baselines: Black-Scholes vs Hull-White vs gradient boosting

Owner: baselines lead.

Sets the yardsticks the neural networks are judged against. Black-Scholes is the floor (gain ratio 0 by definition), Hull-White is the three-parameter regression correction, and LightGBM is a generic learner given the same inputs and loss as the networks.

Run once with `SAMPLE = "main"` and once with `SAMPLE = "traded_only"`; tables and figures go to `tables/<sample>/` and `figures/<sample>/`, predictions to `data/<processed>/predictions/`.

In [ ]:
import sys, pandas as pd, numpy as np, matplotlib.pyplot as plt
sys.path.insert(0, "..")
from config import *
from src import models, evaluate, store

SAMPLE = "main"            # "main" = all quoted options, "traded_only" = volume > 0 (see config.SAMPLES)
P = sample_paths(SAMPLE)
P["tables"].mkdir(parents=True, exist_ok=True); P["figures"].mkdir(parents=True, exist_ok=True)
print(SAMPLE, "->", P["pairs"])

## Load

`store.load_pairs` reads only the modelling columns as float32 (about a fifth of the memory of the full table).

In [ ]:
pairs = store.load_pairs(P["pairs"])
train, val = pairs[pairs.split == "train"], pairs[pairs.split == "val"]
print(f"{len(pairs):,} pairs, {pairs.memory_usage(deep=True).sum() / 1e9:.2f} GB")
pairs.groupby(["split", "type"], observed=True).size().unstack()

## Fit

Both models are fitted on the training years only. LightGBM stops early on the validation year.

In [ ]:
hw = models.HullWhite().fit(train)
gbm = models.GBMResidual(seed=SEED).fit(train, val=val)
print("Hull-White (a, b, c):", {k: np.round(v, 3) for k, v in hw.coef_.items()})
print("LightGBM trees used:", gbm.model_.best_iteration_ or gbm.model_.n_estimators)
gbm.importance()

## Gain ratios

`standard_report` writes five CSVs: by split, by year, by delta bucket and by maturity (test year), and the Feb-Apr 2020 window.

In [ ]:
MODELS = ["hw", "gbm"]
evaluate.add_predictions(pairs, {"hw": hw, "gbm": gbm}, inplace=True)
tables = evaluate.standard_report(pairs, MODELS, P["tables"] / "baselines")
tables["by_split_type"].round(3)

In [ ]:
tables["by_year_type"].round(3)

In [ ]:
tables["crisis"].round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, t in zip(axes, ("C", "P")):
    evaluate.plot_gain_by_bucket(tables["by_delta_bucket"].loc[t], MODELS, f"{'Calls' if t == 'C' else 'Puts'}: test year 2020", ax)
plt.tight_layout(); plt.savefig(P["figures"] / "baselines_gain_by_delta.png", dpi=150)

## How much is there to gain?

Two reference points for reading the numbers above.

**Vega share**: the fraction of the Black-Scholes hedging error that the realised implied-vol change accounts for. It uses information from the end of the hedging interval, so it is a diagnostic, not a hedge. High values (2020) mean the error is mostly about volatility, which is what a delta correction tries to anticipate. Near-zero or negative values (calm years) mean the vendor's day-to-day IV noise outweighs the signal; it is an indicator, not a strict bound.

**In-sample ceiling**: Hull-White refitted on each year and scored on that same year. The gap between the out-of-sample gain and this ceiling is what is lost because the vol-index relationship moves from year to year. It is a ceiling for the Hull-White form only; a richer model can exceed it.

In [ ]:
ctx = evaluate.vega_attribution(pairs, ["year", "type"]).rename(columns={"n": "pairs"})
ctx = ctx.join(evaluate.in_sample_ceiling(pairs, models.HullWhite, "hw"))
ctx = ctx.join(tables["by_year_type"][MODELS].rename(columns=lambda m: f"{m}_out_of_sample"))
ctx.to_csv(P["tables"] / "baselines" / "context_by_year.csv")
ctx.round(3)

## Save predictions

One small file per model; later notebooks join them with `store.load_predictions`.

In [ ]:
store.save_predictions(pairs, MODELS, P["predictions"])

## Findings

*(Write after running: which baseline wins where, how 2019 and 2020 differ, how far each sits below the ceiling, and whether the traded-only sample changes the picture.)*